# MLNogaster hill-climb on the tabular playground dataset

This notebook follows the preprocessing shape in `playground-series-s6e4/preprocess_to_parquet.sql`, concatenates compatible original rows with the synthetic competition train rows, and uses `GAFeatureEngineerDEAP(search_mode="hill_climb")` to search for engineered features that improve MAP@3.

In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import label_binarize

ROOT = Path.cwd()
DATA_DIR = ROOT / "playground-series-s6e4"
sys.path.insert(0, str(ROOT))

from GAfeatureengineer import GAFeatureEngineerDEAP

RANDOM_STATE = 42
TARGET_COL = "Irrigation_Need"
CLASS_LABELS = np.array(["Low", "Medium", "High"])
CLASS_TO_INT = {label: i for i, label in enumerate(CLASS_LABELS)}
INT_TO_CLASS = {i: label for label, i in CLASS_TO_INT.items()}

# Keep this smaller while iterating, then raise it for a longer search.
HC_TRAIN_ROWS = 50_000
HC_POPULATION_SIZE = 80
HC_GENERATIONS = 8
HC_CV_SPLITS = 3

pd.set_option("display.max_columns", 120)


## Metric from the example notebook

In [ ]:
def mapk(actual, predicted, k=3):
    def apk(a, p, k):
        if a in p[:k]:
            return 1.0 / (p[:k].index(a) + 1)
        return 0.0

    return float(np.mean([apk(a, p, k) for a, p in zip(actual, predicted)]))


def map3_from_proba(y_true, proba, classes=None):
    if classes is None:
        classes = np.arange(proba.shape[1])
    classes = np.asarray(classes, dtype=int)
    top3 = np.argsort(proba, axis=1)[:, -3:][:, ::-1]
    predicted = [[INT_TO_CLASS[int(classes[c])] for c in row] for row in top3]
    actual = [INT_TO_CLASS[int(y)] for y in y_true]
    return mapk(actual, predicted, k=3)


def cvs_map3(model, X, y, cv=5, random_state=RANDOM_STATE):
    X = np.asarray(X)
    y = np.asarray(y, dtype=int)
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=random_state)
    scores = []

    for train_idx, val_idx in skf.split(X, y):
        fold_model = clone(model)
        fold_model.fit(X[train_idx], y[train_idx])
        scores.append(map3_from_proba(y[val_idx], fold_model.predict_proba(X[val_idx]), fold_model.classes_))

    return float(np.mean(scores))


## Preprocess and concatenate synthetic + original rows

In [ ]:
RAW_NUMERIC_COLS = [
    "Soil_pH", "Soil_Moisture", "Organic_Carbon", "Electrical_Conductivity",
    "Temperature_C", "Humidity", "Rainfall_mm", "Sunlight_Hours",
    "Wind_Speed_kmh", "Field_Area_hectare", "Mulching_Used",
    "Previous_Irrigation_mm",
]

CATEGORICAL_LEVELS = {
    "Soil_Type": ["Clay", "Loamy", "Sandy", "Silt"],
    "Crop_Type": ["Cotton", "Maize", "Potato", "Rice", "Sugarcane", "Wheat"],
    "Crop_Growth_Stage": ["Flowering", "Harvest", "Sowing", "Vegetative"],
    "Season": ["Kharif", "Rabi", "Zaid"],
    "Irrigation_Type": ["Canal", "Drip", "Rainfed", "Sprinkler"],
    "Water_Source": ["Groundwater", "Rainwater", "Reservoir", "River"],
    "Region": ["Central", "East", "North", "South", "West"],
}


def read_table(path):
    path = Path(path)
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path)


def preprocess_features(df):
    out = pd.DataFrame(index=df.index)
    for col in RAW_NUMERIC_COLS:
        out[col] = pd.to_numeric(df[col], errors="coerce")

    out["Mulching_Used"] = out["Mulching_Used"].fillna(0).astype(int)

    for col, levels in CATEGORICAL_LEVELS.items():
        values = df[col].astype("string")
        for level in levels:
            out[f"{col}_{level}"] = (values == level).astype(int)

    return out


def encode_target(series):
    encoded = series.map(CLASS_TO_INT)
    if encoded.isna().any():
        bad = sorted(series[encoded.isna()].dropna().unique().tolist())
        raise ValueError(f"Unknown target labels: {bad}")
    return encoded.astype(int)


def load_compatible_original_rows(synthetic_columns):
    candidates = [
        DATA_DIR / "original_train.parquet",
        DATA_DIR / "original_train.csv",
        DATA_DIR / "archive" / "train.csv",
        DATA_DIR / "archive" / "trainn.csv",
    ]
    original_frames = []

    for path in candidates:
        if not path.exists():
            continue
        raw = read_table(path)
        required = set(RAW_NUMERIC_COLS) | set(CATEGORICAL_LEVELS) | {TARGET_COL}
        missing = sorted(required - set(raw.columns))
        if missing:
            print(f"Skipping {path.name}: incompatible columns; missing {missing[:8]}")
            continue
        X_orig = preprocess_features(raw).reindex(columns=synthetic_columns)
        y_orig = encode_target(raw[TARGET_COL])
        original_frames.append((path, X_orig, y_orig))

    return original_frames


synthetic_train_raw = read_table(DATA_DIR / "train.parquet").sort_values("id").reset_index(drop=True)
test_raw = read_table(DATA_DIR / "test.parquet").sort_values("id").reset_index(drop=True)
test_ids = test_raw["id"].copy()

X_synth = preprocess_features(synthetic_train_raw)
y_synth = encode_target(synthetic_train_raw[TARGET_COL])
X_test = preprocess_features(test_raw).reindex(columns=X_synth.columns)

original_frames = load_compatible_original_rows(X_synth.columns)
X_parts = [X_synth]
y_parts = [y_synth]

for path, X_orig, y_orig in original_frames:
    print(f"Concatenating original rows from {path}: {len(X_orig):,} rows")
    X_parts.append(X_orig)
    y_parts.append(y_orig)

X_train = pd.concat(X_parts, axis=0, ignore_index=True)
y_train = pd.concat(y_parts, axis=0, ignore_index=True)

print("Synthetic rows:", f"{len(X_synth):,}")
print("Compatible original rows:", f"{sum(len(x) for x in X_parts[1:]):,}")
print("Combined train shape:", X_train.shape)
print("Test shape:", X_test.shape)
display(y_train.value_counts().sort_index().rename(index=INT_TO_CLASS).to_frame("rows"))


## Hill-climb MAP@3 feature search

In [ ]:
def make_hc_metric(cv=HC_CV_SPLITS, random_state=RANDOM_STATE):
    def hc_map3(feature_matrix, y):
        X = np.asarray(feature_matrix, dtype=float)
        y = np.asarray(y, dtype=int)

        model = make_pipeline(
            SimpleImputer(strategy="median"),
            HistGradientBoostingClassifier(
                max_iter=80,
                learning_rate=0.07,
                max_leaf_nodes=15,
                l2_regularization=0.05,
                random_state=random_state,
            ),
        )
        return cvs_map3(model, X, y, cv=cv, random_state=random_state)

    return hc_map3


if HC_TRAIN_ROWS is not None and len(X_train) > HC_TRAIN_ROWS:
    rng = np.random.default_rng(RANDOM_STATE)
    keep_idx = []
    for cls in sorted(y_train.unique()):
        cls_idx = np.flatnonzero(y_train.to_numpy() == cls)
        n_cls = max(1, int(HC_TRAIN_ROWS * len(cls_idx) / len(X_train)))
        keep_idx.extend(rng.choice(cls_idx, size=min(n_cls, len(cls_idx)), replace=False).tolist())
    keep_idx = np.array(sorted(keep_idx))
    X_hc = X_train.iloc[keep_idx].reset_index(drop=True)
    y_hc = y_train.iloc[keep_idx].reset_index(drop=True)
else:
    X_hc = X_train.reset_index(drop=True)
    y_hc = y_train.reset_index(drop=True)

print("Hill-climb search shape:", X_hc.shape)

engineer = GAFeatureEngineerDEAP(
    search_mode="hill_climb",
    hc_metric=make_hc_metric(),
    maximize_hc_metric=True,
    population_size=HC_POPULATION_SIZE,
    generations=HC_GENERATIONS,
    hall_of_fame=12,
    init_min_depth=1,
    init_max_depth=3,
    max_tree_height=6,
    parsimony_coefficient=1e-4,
    early_stop_rounds=4,
    checkpoint_path=str(ROOT / "hc_s6e4_checkpoint.json"),
    checkpoint_every_accepts=1,
    random_state=RANDOM_STATE,
    verbose=True,
)

engineer.fit(X_hc, y_hc)
print("Best hill-climb MAP@3:", engineer.selected_fitness_)
display(engineer.leaderboard_.to_pandas())
engineer.selected_programs_


## Concatenate original columns with engineered columns

In [ ]:
train_hc_all = engineer.transform(X_train).to_pandas()
test_hc_all = engineer.transform(X_test).to_pandas()

# transform() returns the original columns plus selected generated columns.
engineered_cols = [c for c in train_hc_all.columns if c not in X_train.columns]
print("Selected engineered columns:", engineered_cols)

X_train_model = pd.concat(
    [X_train.reset_index(drop=True), train_hc_all[engineered_cols].reset_index(drop=True)],
    axis=1,
)
X_test_model = pd.concat(
    [X_test.reset_index(drop=True), test_hc_all[engineered_cols].reset_index(drop=True)],
    axis=1,
)

print("Model train shape:", X_train_model.shape)
print("Model test shape:", X_test_model.shape)
display(X_train_model.head())


## Cross-validate final model and build submission

In [ ]:
final_model = make_pipeline(
    SimpleImputer(strategy="median"),
    HistGradientBoostingClassifier(
        max_iter=180,
        learning_rate=0.05,
        max_leaf_nodes=31,
        l2_regularization=0.03,
        random_state=RANDOM_STATE,
    ),
)

final_score = cvs_map3(
    final_model,
    X_train_model.to_numpy(dtype=float),
    y_train.to_numpy(dtype=int),
    cv=5,
)
print("Final CV MAP@3:", final_score)


In [ ]:
def make_submission(model, X, y, X_test, test_ids, filename="submission_hc_mlnogaster.csv", n_splits=5):
    X_np = np.asarray(X, dtype=float)
    y_np = np.asarray(y, dtype=int)
    X_test_np = np.asarray(X_test, dtype=float)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    test_preds = np.zeros((X_test_np.shape[0], len(CLASS_LABELS)))
    oof_preds = np.zeros((X_np.shape[0], len(CLASS_LABELS)))

    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_np, y_np), start=1):
        print(f"Training fold {fold_idx}/{n_splits}")
        fold_model = clone(model)
        fold_model.fit(X_np[train_idx], y_np[train_idx])
        fold_oof = fold_model.predict_proba(X_np[val_idx])
        fold_test = fold_model.predict_proba(X_test_np)
        oof_preds[val_idx[:, None], fold_model.classes_.astype(int)] = fold_oof
        test_preds[:, fold_model.classes_.astype(int)] += fold_test / n_splits

    oof_score = map3_from_proba(y_np, oof_preds)
    top3 = np.argsort(test_preds, axis=1)[:, -3:][:, ::-1]
    pred_labels = [[INT_TO_CLASS[int(c)] for c in row] for row in top3]
    submission = pd.DataFrame({
        "id": test_ids.to_numpy(),
        TARGET_COL: [" ".join(row) for row in pred_labels],
    })
    submission.to_csv(filename, index=False)
    print("OOF MAP@3:", oof_score)
    print("Wrote", filename)
    return submission, oof_score


submission, oof_score = make_submission(
    final_model,
    X_train_model,
    y_train,
    X_test_model,
    test_ids,
)
display(submission.head())
